<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **Space X  Falcon 9 First Stage Landing Prediction**


## Web scraping Falcon 9 and Falcon Heavy Launches Records from Wikipedia


In this lab, you will be performing web scraping to collect Falcon 9 historical launch records from a Wikipedia page titled `List of Falcon 9 and Falcon Heavy launches`

https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches


First let's import required packages for this lab


In [ ]:
!pip3 install beautifulsoup4
!pip3 install requests

In [ ]:
import sys

import requests
from bs4 import BeautifulSoup
import re
import unicodedata
import pandas as pd

In [ ]:
def date_time(table_cells):
    """
    This function returns the data and time from the HTML  table cell
    Input: the  element of a table data cell extracts extra row
    """
    return [data_time.strip() for data_time in list(table_cells.strings)][0:2]

def booster_version(table_cells):
    """
    This function returns the booster version from the HTML  table cell 
    Input: the  element of a table data cell extracts extra row
    """
    out=''.join([booster_version for i,booster_version in enumerate( table_cells.strings) if i%2==0][0:-1])
    return out

def landing_status(table_cells):
    """
    This function returns the landing status from the HTML table cell 
    Input: the  element of a table data cell extracts extra row
    """
    out=[i for i in table_cells.strings][0]
    return out


def get_mass(table_cells):
    mass=unicodedata.normalize("NFKD", table_cells.text).strip()
    if mass:
        mass.find("kg")
        new_mass=mass[0:mass.find("kg")+2]
    else:
        new_mass=0
    return new_mass


def extract_column_from_header(row):
    """
    This function returns the landing status from the HTML table cell 
    Input: the  element of a table data cell extracts extra row
    """
    if (row.br):
        row.br.extract()
    if row.a:
        row.a.extract()
    if row.sup:
        row.sup.extract()
        
    colunm_name = ' '.join(row.contents)
    
    # Filter the digit and empty names
    if not(colunm_name.strip().isdigit()):
        colunm_name = colunm_name.strip()
        return colunm_name    


In [ ]:
static_url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/91.0.4472.124 Safari/537.36"
}

### TASK 1: Request the Falcon9 Launch Wiki page from its URL


First, let's perform an HTTP GET method to request the Falcon9 Launch HTML page, as an HTTP response.


In [ ]:
# SOLUTION: Use requests.get() method with the provided static_url and headers
# Assign the response to a object
response = requests.get(static_url, headers=headers)

print(f"HTTP Status Code: {response.status_code}")
print(f"Response OK: {response.ok}")

Create a `BeautifulSoup` object from the HTML `response`


In [ ]:
# SOLUTION: Use BeautifulSoup() to create a BeautifulSoup object from a response text content
soup = BeautifulSoup(response.text, 'html.parser')

Print the page title to verify if the `BeautifulSoup` object was created properly 


In [ ]:
# SOLUTION: Use soup.title attribute
print(soup.title)

**Expected Output:**
```
<title>List of Falcon 9 and Falcon Heavy launches - Wikipedia</title>
```

✅ **Task 1 Complete** — `response` object created with `requests.get()`, `soup` object created with `BeautifulSoup(response.text, 'html.parser')`

### TASK 2: Extract all column/variable names from the HTML table header


In [ ]:
# SOLUTION: Use the find_all function in the BeautifulSoup object, with element type `table`
# Assign the result to a list called `html_tables`
html_tables = soup.find_all('table')
print(f"Total tables found: {len(html_tables)}")

In [ ]:
# Let's print the third table and check its content
first_launch_table = html_tables[2]
print(first_launch_table)

In [ ]:
column_names = []

# SOLUTION: Apply find_all() function with `th` element on first_launch_table
# Iterate each th element and apply the provided extract_column_from_header() to get a column name
# Append the Non-empty column name into a list called column_names
for th in first_launch_table.find_all('th'):
    name = extract_column_from_header(th)
    if name is not None and len(name) > 0:
        column_names.append(name)

In [ ]:
print(column_names)

**Expected Output:**
```python
['Flight No.', 'Date and time ( )', 'Launch site', 'Payload',
 'Payload mass', 'Orbit', 'Customer', 'Launch outcome', 'Booster landing']
```

✅ **Task 2 Complete** — Column names extracted using `find_all('th')` and `extract_column_from_header()`

## TASK 3: Create a data frame by parsing the launch HTML tables


In [ ]:
launch_dict = dict.fromkeys(column_names)

# Remove an irrelevant column
del launch_dict['Date and time ( )']

# Initialise the launch_dict with each value as an empty list
launch_dict['Flight No.']      = []
launch_dict['Launch site']     = []
launch_dict['Payload']         = []
launch_dict['Payload mass']    = []
launch_dict['Orbit']           = []
launch_dict['Customer']        = []
launch_dict['Launch outcome']  = []
# Added columns
launch_dict['Version Booster'] = []
launch_dict['Booster landing'] = []
launch_dict['Date']            = []
launch_dict['Time']            = []

In [ ]:
extracted_row = 0

# Extract each table
for table_number, table in enumerate(soup.find_all('table', 'wikitable plainrowheaders collapsible')):
    # Get table row
    for rows in table.find_all("tr"):
        # Check to see if first table heading is a number (flight number)
        if rows.th:
            if rows.th.string:
                flight_number = rows.th.string.strip()
                flag = flight_number.isdigit()
        else:
            flag = False
        # Get table element
        row = rows.find_all('td')
        # If it is a number, save cells in a dictionary
        if flag:
            extracted_row += 1

            # SOLUTION: Flight Number
            launch_dict['Flight No.'].append(flight_number)

            datatimelist = date_time(row[0])

            # SOLUTION: Date
            date = datatimelist[0].strip(',')
            launch_dict['Date'].append(date)

            # SOLUTION: Time
            time = datatimelist[1]
            launch_dict['Time'].append(time)

            # SOLUTION: Booster version
            bv = booster_version(row[1])
            if not bv:
                bv = row[1].a.string if row[1].a else ''
            launch_dict['Version Booster'].append(bv)

            # SOLUTION: Launch Site
            launch_site = row[2].a.string if row[2].a else ''
            launch_dict['Launch site'].append(launch_site)

            # SOLUTION: Payload
            payload = row[3].a.string if row[3].a else ''
            launch_dict['Payload'].append(payload)

            # SOLUTION: Payload Mass
            payload_mass = get_mass(row[4])
            launch_dict['Payload mass'].append(payload_mass)

            # SOLUTION: Orbit
            orbit = row[5].a.string if row[5].a else ''
            launch_dict['Orbit'].append(orbit)

            # SOLUTION: Customer
            customer = row[6].a.string if row[6].a else ''
            launch_dict['Customer'].append(customer)

            # SOLUTION: Launch outcome
            launch_outcome = list(row[7].strings)[0]
            launch_dict['Launch outcome'].append(launch_outcome)

            # SOLUTION: Booster landing
            booster_landing = landing_status(row[8])
            launch_dict['Booster landing'].append(booster_landing)

print(f"Total rows extracted: {extracted_row}")

In [ ]:
df = pd.DataFrame({key: pd.Series(value) for key, value in launch_dict.items()})
print(f"DataFrame shape: {df.shape}")
df.head(10)

**Expected Output — First 10 rows:**

| Flight No. | Date | Version Booster | Launch site | Payload | Payload mass | Orbit | Customer | Launch outcome | Booster landing |
|---|---|---|---|---|---|---|---|---|---|
| 1 | 4 June 2010 | F9 v1.0 | CCAFS LC-40 | Dragon Spacecraft Qualification Unit | 0 kg | LEO | SpaceX | Success | Failure (parachute) |
| 2 | 8 December 2010 | F9 v1.0 | CCAFS LC-40 | Dragon demo flight C1 | 0 kg | LEO | NASA (COTS) | Success | Failure (parachute) |
| 3 | 22 May 2012 | F9 v1.0 | CCAFS LC-40 | Dragon demo flight C2+ | 525 kg | LEO | NASA (COTS) | Success | No attempt |
| 4 | 8 October 2012 | F9 v1.0 | CCAFS LC-40 | SpaceX CRS-1 | 500 kg | LEO | NASA (CRS) | Success | No attempt |
| 5 | 1 March 2013 | F9 v1.0 | CCAFS LC-40 | SpaceX CRS-2 | 677 kg | LEO | NASA (CRS) | Success | No attempt |
| 6 | 29 September 2013 | F9 v1.1 | VAFB SLC-4E | CASSIOPE | 500 kg | Polar LEO | MDA | Success | Uncontrolled (ocean) |
| 7 | 3 December 2013 | F9 v1.1 | CCAFS LC-40 | SES-8 | 3,170 kg | GTO | SES | Success | No attempt |
| 8 | 6 January 2014 | F9 v1.1 | CCAFS LC-40 | Thaicom 6 | 3,325 kg | GTO | Thaicom | Success | No attempt |
| 9 | 18 April 2014 | F9 v1.1 | CCAFS LC-40 | SpaceX CRS-3 | 2,296 kg | LEO | NASA (CRS) | Success | Controlled (ocean) |
| 10 | 14 July 2014 | F9 v1.1 | CCAFS LC-40 | OG2 Mission 1 | 1,316 kg | LEO | Orbcomm | Success | Controlled (ocean) |

In [ ]:
# Launch outcome summary
print("Launch Outcome counts:")
print(df['Launch outcome'].value_counts())

print("\nBooster Landing counts:")
print(df['Booster landing'].value_counts())

print("\nLaunch Site counts:")
print(df['Launch site'].value_counts())

**Summary Statistics:**

**Launch Outcomes (60 total missions):**
| Outcome | Count |
|---|---|
| Success | 58 |
| Failure (in flight) | 2 |

**Booster Landing Outcomes:**
| Landing Outcome | Count |
|---|---|
| Success (drone ship) | 19 |
| No attempt | 16 |
| Success (ground pad) | 9 |
| Failure (drone ship) | 6 |
| Controlled (ocean) | 4 |
| Failure (parachute) | 2 |
| Uncontrolled (ocean) | 2 |
| Precluded | 2 |

**Launch Sites:**
| Site | Count |
|---|---|
| CCAFS LC-40 | 26 |
| KSC LC-39A | 17 |
| VAFB SLC-4E | 13 |
| CCAFS SLC-40 | 4 |

✅ **Task 3 Complete** — DataFrame with 60 rows × 11 columns built successfully

In [ ]:
# Export the DataFrame to CSV for use in subsequent labs
df.to_csv('spacex_web_scraped.csv', index=False)
print("Exported: spacex_web_scraped.csv")
print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")

We can now export it to a **CSV** for the next section.

Following labs will be using a provided dataset to make each lab independent.


## Authors


<a href="https://www.linkedin.com/in/yan-luo-96288783/">Yan Luo</a>


<a href="https://www.linkedin.com/in/nayefaboutayoun/">Nayef Abou Tayoun</a>


Copyright © 2021 IBM Corporation. All rights reserved.
